In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import scienceplots
import scipy
from dotenv import load_dotenv
import os

import tensorstore as ts

load_dotenv()
PATH = os.getenv("ROOT_PATH")

plt.style.use(['science', 'no-latex'])

def format_ax(ax):
  for spine in ax.spines.values():
    spine.set_linewidth(1.2)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.spines['bottom'].set_visible(False)
  ax.tick_params(which='minor', length=0)
  ax.tick_params(axis='both', labelsize=12)
  for spine in ax.spines.values():
    spine.set_visible(False)
  ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
  ax.tick_params(axis='y', which='both', left=False, right=False, direction="out", width=1.2)

In [ ]:
reference_anat = scipy.io.loadmat(f'{PATH}/Additional_mat_files/ReferenceBrain.mat')

plt.figure(figsize=(10, 10))
plt.imshow(np.sum(reference_anat['anat_stack_norm'], axis=-1).T, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
np.prod(reference_anat['anat_stack_norm'].shape)

In [ ]:
import h5py

f = h5py.File(f'{PATH}/Additional_mat_files/MaskDatabase.mat', 'r')
print(list(f.keys()))
f['MaskDatabase']['ir'][5314833:5320771]

In [ ]:
f['width'][:]

Check mapzebrain and other references

In [ ]:
import nrrd
import nibabel as nib
x, header = nrrd.read('/Users/s/vault/mapzebrain/standard_brain_fixed_SYP_T_GAD1b.nrrd')
y, header = nrrd.read('/Users/s/vault/mapzebrain/Ref20131120pt14pl2.nrrd')

a, _ = nrrd.read('/Users/s/vault/mapzebrain/live_standard_HuClynTagRFP_2.nrrd')
b, _ = nrrd.read('/Users/s/vault/mapzebrain/live_standard_HuClynTagRFP_2.nrrd')
c, _ = nrrd.read('/Users/s/vault/mapzebrain/live_standard_vGlutdsRed.nrrd')

nib_img = nib.load('/Users/s/vault/mapzebrain/huc-gcamp5g-ref-01.nii.gz')
v = nib_img.get_fdata()


In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(np.sum(x, axis=-1), cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(np.sum(y, axis=-1), cmap='gray')
plt.axis('off')

In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(np.sum(y, axis=-1), cmap='gray')
plt.axis('off')

plt.figure(figsize=(10, 10))
plt.imshow(np.sum(c, axis=-1), cmap='gray')
plt.axis('off')

In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(np.sum(y, axis=-1), cmap='gray')
plt.axis('off')
plt.figure(figsize=(10, 10))
plt.imshow(np.sum(v[..., 0], axis=-1).T, cmap='gray')
plt.axis('off')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# Load SWC file
swc_file = '/Users/s/vault/mapzebrain/20150327_1013_BGUG_HuC_ltRFP_d5_F2.swc'
df = pd.read_csv(swc_file, sep=' ', header=None,
                  names=['id', 'type', 'x', 'y', 'z', 'radius', 'parent_id'])

In [ ]:
coordinates = np.stack([df['x'].values, df['y'].values, df['z'].values], -1)
coordinates.shape

In [ ]:
import pyvista as pv
import matplotlib.pyplot as plt
import numpy as np

grid = pv.wrap(x)
grid.spacing = [1.0, 1.0, 1.0]

opacity = [
    0.0,
    0.0,
    0.1,
    0.1,
    0.1,
]

points = coordinates * np.array(grid.spacing)
# colors = np.random.rand(len(points))

point_cloud = pv.PolyData(points)

# Use an off-screen plotter for static image
plotter = pv.Plotter(notebook=True)
plotter.add_mesh(
    point_cloud,
    cmap='magma',
    point_size=10,
    render_points_as_spheres=True,
    show_scalar_bar=False
)

# plotter.add_volume(
#     grid,
#     cmap="gray",
#     opacity=opacity,
#     shade=True,
#     show_scalar_bar=False,
# )

plotter.camera_position = 'xy'
plotter.camera.azimuth = 0
plotter.camera.elevation = 0
plotter.camera.zoom(1.0)
plotter.show()

# Render and save to a temporary file, then display as static image
# img = plotter.screenshot(return_img=True)
# plotter.close()

# plt.figure(figsize=(8, 8), dpi=1500)
# plt.imshow(img)
# plt.axis('off')
# plt.show()

In [ ]:
# Let's create a combined visualization to compare morphology vs point cloud
import neurom as nm
from neurom.view import plot_morph3d
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# Load data
swc_file = '/Users/s/vault/mapzebrain/20150327_1013_BGUG_HuC_ltRFP_d5_F2.swc'
morphology = nm.load_morphology(swc_file)
df = pd.read_csv(swc_file, sep=' ', header=None,
                 names=['id', 'type', 'x', 'y', 'z', 'radius', 'parent_id'])

# Extract coordinates from SWC
coordinates = np.stack([df['x'].values, df['y'].values, df['z'].values], -1)

print(f"SWC data shape: {coordinates.shape}")
print(f"Coordinate ranges:")
print(f"  X: {coordinates[:, 0].min():.1f} to {coordinates[:, 0].max():.1f}")
print(f"  Y: {coordinates[:, 1].min():.1f} to {coordinates[:, 1].max():.1f}")
print(f"  Z: {coordinates[:, 2].min():.1f} to {coordinates[:, 2].max():.1f}")

# Create side-by-side comparison
fig = plt.figure(figsize=(20, 8))

# Left plot: NeuroM morphology with connections
ax1 = fig.add_subplot(121, projection='3d')
plot_morph3d(morphology, ax=ax1)
ax1.set_title('NeuroM Morphology\n(with connections)', fontsize=14)
ax1.set_xlabel('X (μm)')
ax1.set_ylabel('Y (μm)')
ax1.set_zlabel('Z (μm)')

# Right plot: Point cloud (your PyVista approach)
ax2 = fig.add_subplot(122, projection='3d')

# Color points by type
type_colors = {1: 'red', 2: 'blue', 3: 'green', 4: 'orange', 5: 'purple', 6: 'brown'}
for neuron_type in df['type'].unique():
    mask = df['type'] == neuron_type
    ax2.scatter(df[mask]['x'], df[mask]['y'], df[mask]['z'],
              c=type_colors.get(neuron_type, 'black'),
              s=df[mask]['radius']*5, alpha=0.8,
              label=f'Type {neuron_type}')

ax2.set_title('Point Cloud\n(SWC points only)', fontsize=14)
ax2.set_xlabel('X (μm)')
ax2.set_ylabel('Y (μm)')
ax2.set_zlabel('Z (μm)')
ax2.legend()

# Make axes equal for comparison
for ax in [ax1, ax2]:
    ax.set_xlim([coordinates[:, 0].min()-10, coordinates[:, 0].max()+10])
    ax.set_ylim([coordinates[:, 1].min()-10, coordinates[:, 1].max()+10])
    ax.set_zlim([coordinates[:, 2].min()-10, coordinates[:, 2].max()+10])

plt.tight_layout()
plt.show()

# Check if morphology matches point cloud
print(f"\nComparison:")
print(f"NeuroM morphology center: {morphology.soma.center}")
print(f"Point cloud center: {coordinates.mean(axis=0)}")
print(f"Total points in SWC: {len(coordinates)}")
print(f"Morphology sections: {nm.get('number_of_sections', morphology)}")

# Extract actual coordinates from NeuroM morphology for direct comparison
neurom_points = []
for neurite in morphology.neurites:
    for section in neurite.iter():
        neurom_points.extend(section.points)

neurom_coords = np.array(neurom_points)
print(f"NeuroM extracted points: {len(neurom_coords)}")
print(f"Coordinate match check (first 5 points):")
print("SWC coords:", coordinates[:5])
print("NeuroM coords:", neurom_coords[:5] if len(neurom_coords) > 0 else "No points extracted")

In [ ]:
import h5py
import numpy as np

f = h5py.File(f'{PATH}/Additional_mat_files/MaskDatabase.mat', 'r')
names = f['MaskDatabaseNames']
print([(i, "".join([chr(c[0]) for c in f[name[0]]])) for i, name in enumerate(names)])

In [ ]:
def linear_to_3d_matlab(linear_idx, width, height):
    idx = linear_idx - 1
    i = idx % width
    j = (idx // width) % height
    k = idx // (width * height)
    return i, j, k

In [ ]:
ir = f['MaskDatabase']['ir'][:]  # Row indices (linear voxel indices)
jc = f['MaskDatabase']['jc'][:]  # Column pointers
data = f['MaskDatabase']['data'][:]  # Should be all ones

mask_idx = 77
start_idx = jc[mask_idx]
end_idx = jc[mask_idx + 1]

In [ ]:
mask = np.zeros_like(reference_anat['anat_stack_norm'])
mask_shape = mask.shape
for i in ir[start_idx:end_idx]:
  i_x, i_y, i_z = linear_to_3d_matlab(i, mask_shape[0], mask_shape[1])
  mask[i_x, i_y, i_z]=1

In [ ]:
anat_proj = np.sum(reference_anat['anat_stack_norm'], -1).T
mask_proj = np.sum(mask, -1).T

plt.figure(figsize=(10, 10))
plt.imshow(anat_proj, cmap='gray')
plt.imshow(mask_proj, cmap='Reds', alpha=0.4)
plt.axis('off')
plt.show()

Max Planck masks

In [ ]:
os.listdir('/Users/s/Downloads/mapZebrain__regions__v2.0.1')

In [ ]:
# Try reading MaskDatabaseNames as object references
print("Exploring MaskDatabaseNames structure:")
names_data = f['MaskDatabaseNames']
print(f"  Shape: {names_data.shape}")
print(f"  Dtype: {names_data.dtype}")

# Try reading as references
mask_names = []
for i in range(n_masks):
    # Try accessing as object reference
    ref = names_data[i, 0]
    if isinstance(ref, h5py.h5r.Reference):
        name_obj = f[ref]
        name = ''.join(chr(c[0]) for c in name_obj[:])
    else:
        # Try direct dereferencing
        name_obj = f[ref]
        name = ''.join(chr(c[0]) for c in name_obj[:])
    mask_names.append(name)

In [ ]:
mask_names

Registration with ANTs

In [ ]:
import ants
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

target = np.zeros((100, 100), dtype=np.float32)
target[30:70, 30:70] = 1.0

moving = np.zeros((100, 100), dtype=np.float32)
moving[40:80, 20:60] = 1.0

coords_moving = np.random.rand(10, 2) * 40 + np.array([20, 40])

target_img = ants.from_numpy(target)
moving_img = ants.from_numpy(moving)

result = ants.registration(fixed=target_img, moving=moving_img, type_of_transform='SyN')

warped = result['warpedmovout']

coords_df = pd.DataFrame(coords_moving, columns=['x', 'y'])

coords_registered = ants.apply_transforms_to_points(
    dim=2,
    points=coords_df,
    transformlist=result['fwdtransforms']
)

coords_registered_np = coords_registered[['x', 'y']].values

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(moving, cmap='gray', origin='lower')
axes[0].scatter(coords_moving[:, 0], coords_moving[:, 1], c='red', s=100, marker='x')
axes[0].set_title('Moving Image + Original Coords')
axes[1].imshow(warped.numpy(), cmap='gray', origin='lower')
axes[1].scatter(coords_registered_np[:, 0], coords_registered_np[:, 1], c='lime', s=100, marker='x')
axes[1].set_title('Registered Image + Transformed Coords')
plt.tight_layout()
plt.show()

In [ ]:
x_2 = np.sum(x, axis=-1)
y_2 = np.sum(y, axis=-1)
x_2_red = skimage.measure.block_reduce(x_2, (4, 4))
y_2_red = skimage.measure.block_reduce(y_2, (4, 4))

In [ ]:
target_img = ants.from_numpy(x_2_red)
moving_img = ants.from_numpy(y_2_red)

result = ants.registration(
    fixed=target_img,
    moving=moving_img,
    type_of_transform='Affine',
    aff_metric='MI',
    aff_sampling=32,
    aff_random_sampling_rate=0.25,
    aff_iterations=(200, 200, 200, 0),
    aff_shrink_factors=(12, 8, 4, 2),
    aff_smoothing_sigmas=(4, 3, 2, 1),
    syn_metric='CC',
    syn_sampling=2,
    reg_iterations=(200, 200, 200, 200, 10),
    grad_step=0.05,
    flow_sigma=6,
    total_sigma=0.5,
    verbose=True
)

warped = result['warpedmovout']

# coords_df = pd.DataFrame(coords_moving, columns=['x', 'y'])

# coords_registered = ants.apply_transforms_to_points(
#     dim=2,
#     points=coords_df,
#     transformlist=result['fwdtransforms']
# )

# coords_registered_np = coords_registered[['x', 'y']].values

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(target_img.numpy(), cmap='gray', origin='lower')
# axes[0].scatter(coords_moving[:, 0], coords_moving[:, 1], c='red', s=100, marker='x')
axes[0].set_title('Moving Image + Original Coords')
axes[1].imshow(warped.numpy(), cmap='gray', origin='lower')
# axes[1].scatter(coords_registered_np[:, 0], coords_registered_np[:, 1], c='lime', s=100, marker='x')
axes[1].set_title('Registered Image + Transformed Coords')
plt.tight_layout()
plt.show()

In [ ]:
import ants
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

np.random.seed(42)

moving_mask = np.random.rand(100, 100) > 0.7
moving_mask = gaussian_filter(moving_mask.astype(float), sigma=3) > 0.3
moving_mask = np.roll(moving_mask, shift=(10, -10), axis=(0, 1)).astype(np.float32)

target_mask = np.random.rand(100, 100) > 0.7
target_mask = gaussian_filter(target_mask.astype(float), sigma=3) > 0.3
target_mask = target_mask.astype(np.float32)

coords_moving = np.random.rand(10, 2) * 60 + 20

moving_img = ants.from_numpy(moving_mask)
target_img = ants.from_numpy(target_mask)

result = ants.registration(
    fixed=target_img,
    moving=moving_img,
    type_of_transform='SyN',
    aff_metric='meansquares',
    syn_metric='meansquares',
    reg_iterations=(100, 70, 50, 20)
)

warped = result['warpedmovout']

coords_df = pd.DataFrame(coords_moving, columns=['x', 'y'])
coords_registered = ants.apply_transforms_to_points(dim=2, points=coords_df, transformlist=result['fwdtransforms'])
coords_registered_np = coords_registered[['x', 'y']].values

fig, axes = plt.subplots(1, 3, figsize=(12, 5))
axes[0].imshow(target_mask, cmap='gray', origin='lower')
axes[1].imshow(moving_mask, cmap='gray', origin='lower')
axes[1].scatter(coords_moving[:, 0], coords_moving[:, 1], c='red', s=100, marker='x')
axes[1].set_title('Moving Mask + Original Coords')
axes[2].imshow(warped.numpy(), cmap='gray', origin='lower')
axes[2].scatter(coords_registered_np[:, 0], coords_registered_np[:, 1], c='lime', s=100, marker='x')
axes[2].set_title('Registered Mask + Transformed Coords')
plt.tight_layout()
plt.show()